# CSRBB — Credit Spread Risk in the Banking Book

EBA/GL/2022/14 — identification, CS01 sensitivity, and scenario stress P&L.

CSRBB covers the risk that banking-book positions lose value due to credit
spread movements that are not driven by the institution's own credit standing
or by general interest rate risk. Unlike FRTB CSR (trading book), there is
no prescriptive capital formula — the standard management metrics are
**CS01** (spread duration sensitivity) and **stress P&L** under parallel shocks.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from banking_risk.csrbb import (
    Spread_Position,
    SA_CSRBB_Calculator,
    CSRBB_STRESS_SCENARIOS,
)
from banking_risk.utils.reporting import Dark_Style

## 2. Portfolio

A representative banking-book credit portfolio: senior bonds and loans
across the rating spectrum. Spreads are set to stylised market levels
(investment grade: 50–150 bps; speculative grade: 200–400 bps).

In [ ]:
positions = [
    Spread_Position("Corp_Bond_A_5Y",     "EUR", 50_000_000,  5.0, 0.0060, rating="A"),
    Spread_Position("Corp_Bond_BBB_7Y",   "EUR", 80_000_000,  7.0, 0.0110, rating="BBB"),
    Spread_Position("Corp_Bond_BBB_10Y",  "EUR", 60_000_000, 10.0, 0.0130, rating="BBB"),
    Spread_Position("Corp_Bond_BB_3Y",    "EUR", 30_000_000,  3.0, 0.0220, rating="BB"),
    Spread_Position("Corp_Loan_BB_5Y",    "EUR", 40_000_000,  5.0, 0.0280, rating="BB"),
    Spread_Position("Sub_Bond_B_3Y",      "EUR", 20_000_000,  3.0, 0.0400, rating="B"),
]

print(f"Positions  : {len(positions)}")
print(f"Total notional: EUR {sum(p.notional for p in positions):>15,.0f}")

## 3. CS01 and base PV

CS01 = notional × maturity × disc × 1bp — the change in portfolio value
for a 1bp parallel widening of all credit spreads.

Formula (zero-coupon approximation):
> PV  = N × exp(−(r_f + s) × T)
> CS01 = N × T × exp(−(r_f + s) × T) × 10⁻⁴

In [ ]:
calc   = SA_CSRBB_Calculator(risk_free_rate=0.03)
result = calc.compute(positions)

display = result.detail[["notional", "maturity_years", "z_spread_bps", "rating", "pv", "cs01"]].copy()
display["pv_M"]   = display["pv"]   / 1e6
display["cs01_K"] = display["cs01"] / 1e3
display[["notional", "maturity_years", "z_spread_bps", "rating", "pv_M", "cs01_K"]].round(3)

In [ ]:
print(f"Total CS01     : EUR {result.total_cs01:>10,.2f}  (per 1bp parallel widening)")
print(f"Total PV       : EUR {result.detail['pv'].sum():>15,.0f}")
print(f"DV01 / PV      : {result.total_cs01 / result.detail['pv'].sum() * 1e4:.4f} %  (spread duration proxy)")

## 4. CS01 by rating bucket

Regulatory CSRBB reporting (EBA/GL/2022/14 Annex I) requires CS01
broken down by credit quality step / rating bucket.

In [ ]:
rating_cs01 = pd.Series(result.cs01_by_rating, name="CS01").sort_index()
print(rating_cs01.to_string())

Dark_Style().apply()
p = Dark_Style().palette

fig, ax = plt.subplots(figsize=(8, 4))
colors = [p["cyan"] if v >= 0 else p["red"] for v in rating_cs01]
ax.bar(rating_cs01.index, rating_cs01.values / 1e3, color=colors, alpha=0.85, width=0.5)
ax.set_ylabel("CS01 (EUR thousands per 1bp)")
ax.set_title("CSRBB — CS01 by rating bucket", color=p["text_title"], fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Scenario stress P&L

Full closed-form revaluation under parallel spread shocks.
Default scenarios: +100 bps (spread widening) and −50 bps (tightening).

> ΔPV(Δs) = Σᵢ Nᵢ × [exp(−(r_f + sᵢ + Δs) × Tᵢ) − exp(−(r_f + sᵢ) × Tᵢ)]

In [ ]:
stress = pd.Series(result.stress_pnl, name="Stress P&L (EUR)")
stress_M = stress / 1e6

print("Stress P&L:")
for scenario, pnl in stress.items():
    print(f"  {scenario:<25}: EUR {pnl:>12,.0f}")

print(f"\nCS01 × 100  : EUR {-result.total_cs01 * 100:>12,.0f}  (linear approx)")
print(f"Full reprice: EUR {stress['spread_widen_100bp']:>12,.0f}  (convexity included)")

In [ ]:
Dark_Style().apply()
p = Dark_Style().palette

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: CS01 per position
cs01_k = result.detail["cs01"] / 1e3
colors = [p["cyan"] if v >= 0 else p["red"] for v in cs01_k]
ax1.barh(result.detail.index, cs01_k, color=colors, alpha=0.85)
ax1.set_xlabel("CS01 (EUR thousands per 1bp)")
ax1.set_title("CS01 per position", color=p["text_title"])
ax1.grid(axis="x", alpha=0.3)

# Panel 2: Stress P&L
scenarios = list(stress_M.index)
s_colors  = [p["red"] if v < 0 else p["green"] for v in stress_M]
ax2.bar(scenarios, stress_M.values, color=s_colors, alpha=0.85, width=0.4)
ax2.axhline(0, color=p["text_muted"], lw=0.8, ls="--")
ax2.set_ylabel("Stress P&L (EUR millions)")
ax2.set_title("Stress P&L by scenario", color=p["text_title"])
ax2.tick_params(axis="x", rotation=15)
ax2.grid(axis="y", alpha=0.3)

fig.suptitle("CSRBB — CS01 and Scenario Stress (EBA/GL/2022/14)",
             color=p["text_title"], fontweight="bold")
fig.tight_layout()
plt.show()

## 6. Custom stress scenarios

Pass any `dict[str, float]` (scenario name → bps) to apply institution-specific
or regulator-requested shocks.

In [ ]:
custom_scenarios = {
    "mild_widen_50bp"  :  50.0,
    "severe_widen_200bp": 200.0,
    "severe_tighten_100bp": -100.0,
}
custom_calc   = SA_CSRBB_Calculator(scenarios=custom_scenarios, risk_free_rate=0.03)
custom_result = custom_calc.compute(positions)

pd.Series(custom_result.stress_pnl, name="Stress P&L (EUR)").map(
    lambda x: f"EUR {x:>12,.0f}"
).to_frame()